# ③ 검증 — 평가 데이터셋
담당: 검증 · 코드: `rag/validation/validation.py`

증강 Golden dev 396 · v12 공유 평가 패키지(`Golden_Dev396_v12_Team_Package`, ZIP 을 `data/` 에 그대로 푼 폴더 — `config.yaml` 의 `validation.package`)의
`data/golden_dev.jsonl` 396문항과 기존 골든 88문항(`eval/gold_dev.jsonl` 55 · `eval/gold_needle.jsonl` 33 — 두 문서 비교 7문항 포함)을 평가셋으로 씁니다(`validation.golden_sets`).
질문 · 정답(`reference`) · 핵심 사실(`key_facts`)은 바꾸지 않고 그대로 옮겨 점검한 뒤 `artifacts/eval_set.jsonl` 에 저장합니다 → 04 생성이 답하고 05 평가가 채점합니다.
484문항 모두 `reset_context = true` 라 04 생성은 질문마다 새 대화로 답합니다(이전 질문 · 답변을 보내지 않아 토큰이 늘지 않음).
05 결과에서 문항 id 앞부분으로 셋을 가를 수 있습니다 — 패키지 `numeric-…` · `table-…` 등, `gold_dev` 는 `refresh50-…` · `cutover-…`, `gold_needle` 은 `needle-…`.
검색 · 생성 · API 호출은 없습니다.

In [1]:
import pandas as pd

from rag.config import ARTIFACTS, ROOT, load_config, write_jsonl
from rag.validation.validation import check_eval_items, load_eval_items, package_evidence_summary

cfg = load_config()
package_dir = ROOT / cfg["validation"]["package"]

## 평가 항목 점검 · 저장
질문 · 정답 · 핵심 사실이 비었거나 같은 질문이 두 번 나오는 항목은 사유와 함께 빼고, 통과한 항목만 저장합니다. 484문항(패키지 396 + 기존 골든 88)이 모두 통과해야 합니다. 아래 첫 표는 파일별 문항 수입니다.

In [2]:
items = load_eval_items(cfg["validation"]["golden_sets"])
display(pd.Series({path: len(load_eval_items([path])) for path in cfg["validation"]["golden_sets"]}, name="문항"))
eval_set, report = check_eval_items(items)
write_jsonl(eval_set, ARTIFACTS / "eval_set.jsonl")
print(f"평가 데이터셋 {len(eval_set)}개 / 점검 {len(items)}개")
display(report["점검"].value_counts())
display(report[report["점검"] != "통과"])

data/Golden_Dev396_v12_Team_Package/data/golden_dev.jsonl    396
eval/gold_dev.jsonl                                           55
eval/gold_needle.jsonl                                        33
Name: 문항, dtype: int64

평가 데이터셋 484개 / 점검 484개


점검
통과    484
Name: count, dtype: int64

,id,질문,점검


## 질문 유형별 근거 보존 판정 (패키지 `evidence/question_status.json`)
패키지 396문항에 대해 패키지가 원문 → 청크 경로로 근거를 확인한 결과입니다(기존 골든 88문항은 판정이 없어 이 표에 없음). **검색 성능 점수가 아닙니다** — 05 평가 결과를 나눠 읽을 때 씁니다.
- `F` (199): 핵심 근거가 원문 → 청크 경로로 입증된 문항
- `D` (194): 표 · 조건 · 분산 근거라 추가 확인이 필요한 문항 (데이터 누락이라는 뜻은 아님)
- `B` (3): 골든 자체 검토 후보 — `table-A-fill2-r1-0059`, `table-A-fill4-r2-0077`, `comparison-A-fill3-r2-0077`. 정답은 고치지 않았으며 결과를 따로 보고합니다.

`comparison` 56문항은 모두 문서 하나 안의 비교입니다(두 문서 비교 아님).

In [3]:
package_evidence_summary(package_dir)

판정,F,D,B,전체
질문 유형,,,,
comparison,17,38,1,56
condition,16,53,0,69
multi_hop,1,70,0,71
numeric,78,26,0,104
table,87,7,2,96
전체,199,194,3,396
